# 00 — Dynamic Google Colab Setup

This notebook prepares the environment for the FIQA demographic analysis project **without requiring a fixed Google Drive folder or a fixed checkpoint filename**.

The user selects:

- the project directory,
- the DiveFace image directory,
- the annotation file,
- any available CR-FIQA checkpoint (`.pth`, `.pt`, or `.ckpt`),
- and optionally a different CR-FIQA source repository.

The selected paths are validated and saved to a JSON configuration file so that later scripts can reuse them.


In [ ]:
# Mount Google Drive and import setup utilities

from pathlib import Path
import json
import subprocess
import sys

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is intended to run in Google Colab."
    ) from exc

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f"Google Drive root not found: {DRIVE_ROOT}")

print(f"Google Drive mounted: {DRIVE_ROOT}")


In [ ]:
# Helper functions for interactive path selection

def resolve_user_path(value: str, base: Path = DRIVE_ROOT) -> Path:
    """Resolve an absolute path or a path relative to Google Drive/MyDrive."""
    value = value.strip().strip('"').strip("'")
    path = Path(value).expanduser()

    if not path.is_absolute():
        path = base / path

    return path.resolve()


def ask_existing_path(
    prompt: str,
    *,
    base: Path = DRIVE_ROOT,
    expected_type: str = "any",
) -> Path:
    """Ask until the user provides an existing file or directory."""
    while True:
        candidate = resolve_user_path(input(prompt), base=base)

        if not candidate.exists():
            print(f"Not found: {candidate}")
            continue

        if expected_type == "file" and not candidate.is_file():
            print(f"Expected a file, but received: {candidate}")
            continue

        if expected_type == "directory" and not candidate.is_dir():
            print(f"Expected a directory, but received: {candidate}")
            continue

        return candidate


def choose_from_candidates(
    candidates,
    title: str,
    *,
    allow_manual_path: bool = True,
    base: Path = DRIVE_ROOT,
    expected_type: str = "file",
) -> Path:
    """Let the user choose a discovered path or enter another path manually."""
    candidates = sorted({Path(path).resolve() for path in candidates})

    print(f"\n{title}")

    if candidates:
        for index, path in enumerate(candidates, start=1):
            print(f"[{index}] {path}")

        if allow_manual_path:
            print("[M] Enter a different path manually")

        while True:
            choice = input("Selection: ").strip()

            if choice.lower() == "m" and allow_manual_path:
                return ask_existing_path(
                    "Enter path: ",
                    base=base,
                    expected_type=expected_type,
                )

            if choice.isdigit():
                position = int(choice)
                if 1 <= position <= len(candidates):
                    return candidates[position - 1]

            print("Invalid selection. Please try again.")

    return ask_existing_path(
        "No candidates were found. Enter path manually: ",
        base=base,
        expected_type=expected_type,
    )


In [ ]:
# Select the project directory dynamically

print(
    "Enter the project folder path. You may use either:\n"
    "- an absolute path, e.g. /content/drive/MyDrive/FIQA_TEST\n"
    "- or a path relative to MyDrive, e.g. FIQA_TEST"
)

PROJECT_PATH = ask_existing_path(
    "Project directory: ",
    expected_type="directory",
)

print(f"Selected project directory: {PROJECT_PATH}")


In [ ]:
# Discover and select the image directory

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

candidate_image_directories = []

for directory in [PROJECT_PATH, *PROJECT_PATH.rglob("*")]:
    if not directory.is_dir():
        continue

    try:
        contains_images = any(
            child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS
            for child in directory.iterdir()
        )
    except PermissionError:
        continue

    if contains_images:
        candidate_image_directories.append(directory)

IMAGE_FOLDER = choose_from_candidates(
    candidate_image_directories,
    "Select the directory that directly contains DiveFace images "
    "or one of its image-containing subdirectories.",
    expected_type="directory",
    base=PROJECT_PATH,
)

image_count = sum(
    1
    for path in IMAGE_FOLDER.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if image_count == 0:
    raise FileNotFoundError(
        f"No supported image files were found below: {IMAGE_FOLDER}"
    )

print(f"Selected image directory: {IMAGE_FOLDER}")
print(f"Images found recursively: {image_count}")


In [ ]:
# Discover and select the annotation file

annotation_candidates = [
    path
    for pattern in ("*.pkl", "*.pickle", "*.csv", "*.parquet")
    for path in PROJECT_PATH.rglob(pattern)
    if path.is_file()
]

ANNOTATION_FILE = choose_from_candidates(
    annotation_candidates,
    "Select the annotation file.",
    expected_type="file",
    base=PROJECT_PATH,
)

print(f"Selected annotation file: {ANNOTATION_FILE}")


In [ ]:
# Discover and select any available backbone/checkpoint file

checkpoint_candidates = [
    path
    for pattern in ("*.pth", "*.pt", "*.ckpt")
    for path in PROJECT_PATH.rglob(pattern)
    if path.is_file()
]

MODEL_PATH = choose_from_candidates(
    checkpoint_candidates,
    "Select the CR-FIQA backbone/checkpoint to use. "
    "No specific filename is required.",
    expected_type="file",
    base=PROJECT_PATH,
)

print(f"Selected checkpoint: {MODEL_PATH}")


In [ ]:
# Configure the external CR-FIQA source repository

default_repository_url = "https://github.com/fdbtrs/CR-FIQA.git"
repository_input = input(
    "CR-FIQA repository URL "
    f"[press Enter for {default_repository_url}]: "
).strip()

CR_FIQA_REPOSITORY_URL = repository_input or default_repository_url
CR_FIQA_PATH = Path("/content/CR-FIQA")

print(f"CR-FIQA repository: {CR_FIQA_REPOSITORY_URL}")
print(f"Temporary clone path: {CR_FIQA_PATH}")


In [ ]:
# Validate the selected inputs

required_paths = {
    "Project directory": PROJECT_PATH,
    "DiveFace image folder": IMAGE_FOLDER,
    "Annotation file": ANNOTATION_FILE,
    "CR-FIQA checkpoint": MODEL_PATH,
}

errors = []

for name, path in required_paths.items():
    if not path.exists():
        errors.append(f"{name} does not exist: {path}")

if not PROJECT_PATH.is_dir():
    errors.append(f"Project path is not a directory: {PROJECT_PATH}")

if not IMAGE_FOLDER.is_dir():
    errors.append(f"Image path is not a directory: {IMAGE_FOLDER}")

if not ANNOTATION_FILE.is_file():
    errors.append(f"Annotation path is not a file: {ANNOTATION_FILE}")

if not MODEL_PATH.is_file():
    errors.append(f"Checkpoint path is not a file: {MODEL_PATH}")

if MODEL_PATH.suffix.lower() not in {".pth", ".pt", ".ckpt"}:
    errors.append(
        "Checkpoint must normally use one of these extensions: "
        ".pth, .pt, .ckpt"
    )

if errors:
    raise ValueError(
        "Configuration validation failed:\n- " + "\n- ".join(errors)
    )

print("All selected project inputs are valid.")


In [ ]:
# Save the runtime configuration for later scripts

RUNTIME_CONFIG = {
    "project_path": str(PROJECT_PATH),
    "image_folder": str(IMAGE_FOLDER),
    "annotation_file": str(ANNOTATION_FILE),
    "model_path": str(MODEL_PATH),
    "cr_fiqa_path": str(CR_FIQA_PATH),
    "cr_fiqa_repository_url": CR_FIQA_REPOSITORY_URL,
}

RUNTIME_CONFIG_PATH = Path("/content/fiqa_runtime_config.json")
PROJECT_CONFIG_PATH = PROJECT_PATH / "fiqa_runtime_config.json"

config_text = json.dumps(RUNTIME_CONFIG, indent=2)

RUNTIME_CONFIG_PATH.write_text(config_text, encoding="utf-8")
PROJECT_CONFIG_PATH.write_text(config_text, encoding="utf-8")

print(f"Runtime configuration saved to: {RUNTIME_CONFIG_PATH}")
print(f"Drive configuration saved to:   {PROJECT_CONFIG_PATH}")
print("\nConfiguration:")
print(config_text)


In [ ]:
# Clone or update CR-FIQA and install dependencies

if not CR_FIQA_PATH.exists():
    print("Cloning the CR-FIQA repository...")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            CR_FIQA_REPOSITORY_URL,
            str(CR_FIQA_PATH),
        ],
        check=True,
    )
else:
    print("CR-FIQA repository already exists in this runtime.")
    subprocess.run(
        [
            "git",
            "-C",
            str(CR_FIQA_PATH),
            "pull",
            "--ff-only",
        ],
        check=True,
    )

required_packages = [
    "tensorboard",
    "easydict",
    "scikit-learn",
]

print("Installing required packages...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *required_packages,
    ],
    check=True,
)

print("Dependencies installed successfully.")


In [ ]:
# Environment summary

import sklearn
import torch

print("===== Environment Check =====")
print(f"Project directory:    {PROJECT_PATH}")
print(f"Image folder:         {IMAGE_FOLDER}")
print(f"Images found:         {image_count}")
print(f"Annotation file:      {ANNOTATION_FILE}")
print(f"Model checkpoint:     {MODEL_PATH}")
print(f"CR-FIQA repository:   {CR_FIQA_PATH}")
print(f"Configuration file:   {RUNTIME_CONFIG_PATH}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"PyTorch version:      {torch.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:                  {torch.cuda.get_device_name(0)}")
else:
    print("Warning: No GPU is active. CR-FIQA inference may be slow.")

print("\nDynamic setup completed successfully.")
